In [1]:
import os
import time
import copy
import random
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset

In [2]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

def reset_model(model, base_state, device):
    model.load_state_dict(copy.deepcopy(base_state))
    model.to(device)
    return model

import torch
import torch.nn as nn
import torch.optim as optim


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return 100.0 * correct / total

def gradient_ascent_unlearning(
    model,
    forget_train_dl,
    device,
    lr=1e-5,
    epochs=5):
    
    """
    Gradient Ascent unlearning on forget set only.
    """

    model = model.to(device)
    model.train()

    optimizer = optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        running_loss = 0.0

        for images, labels in forget_train_dl:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # NEGATIVE loss => gradient ascent
            loss = -criterion(model(images), labels)
            loss.backward()

            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(forget_train_dl)
        print(
            f"[GA] Epoch {epoch+1}/{epochs} | "
            f"Avg Ascent Loss: {avg_loss:.4f}"
        )

    return model

In [3]:
from torch.nn import functional as F
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

def entropy(p, dim=-1, keepdim=False):
    return -torch.where(p > 0, p * p.log(), p.new([0.0])).sum(dim=dim, keepdim=keepdim)


def collect_prob(data_loader, model):
    data_loader = torch.utils.data.DataLoader(
        data_loader.dataset, batch_size=1, shuffle=False
    )
    prob = []
    with torch.no_grad():
        for batch in data_loader:
            batch = [tensor.to(next(model.parameters()).device) for tensor in batch]
            data, target = batch
            output = model(data)
            prob.append(F.softmax(output, dim=-1).data)
    return torch.cat(prob)


# https://arxiv.org/abs/2205.08096
def get_membership_attack_data(retain_loader, forget_loader, test_loader, model):
    retain_prob = collect_prob(retain_loader, model)
    forget_prob = collect_prob(forget_loader, model)
    test_prob = collect_prob(test_loader, model)

    n_test = len(test_prob)
    idx = torch.randperm(len(retain_prob))[:n_test]
    retain_prob_balanced = retain_prob[idx]

    X_r = (
        torch.cat([entropy(retain_prob_balanced), entropy(test_prob)])
        .cpu()
        .numpy()
        .reshape(-1, 1)
    )
    Y_r = np.concatenate([np.ones(len(retain_prob_balanced)), np.zeros(len(test_prob))])

    X_f = entropy(forget_prob).cpu().numpy().reshape(-1, 1)
    Y_f = np.concatenate([np.ones(len(forget_prob))])
    return X_f, Y_f, X_r, Y_r


# https://arxiv.org/abs/2205.08096
def get_membership_attack_prob(retain_loader, forget_loader, test_loader, model):
    X_f, Y_f, X_r, Y_r = get_membership_attack_data(
        retain_loader, forget_loader, test_loader, model
    )
    
    threshold = X_r[Y_r == 1].mean()  # mean entropy of retain
    results = (X_f < threshold).astype(int)
    
    return results.mean()

In [7]:
class UncertaintyCurriculumOrderer:
    def __init__(self, device=None, seed=42):
        set_seed(seed)
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        self.model = None
        self.train_ds = None
        self.test_ds = None

        self.forget_train_dataset = None
        self.retain_train_dataset = None
        self.forget_test_dataset = None
        self.retain_test_dataset = None

        self.train_dl = None
        self.test_dl = None

        self.uncertainties = None  # aligned with forget_train_dataset

    # ---------------------------------------------------------
    # DATA
    # ---------------------------------------------------------
    def load_data(self, batch_size=256, forget_class=0, num_workers=4):
        transform_train = transforms.Compose([
            transforms.Resize(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010)
            )
        ])
        
        transform_test = transforms.Compose([
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize(
                (0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010)
            )
        ])
        self.train_ds = torchvision.datasets.CIFAR10(
            root='/kaggle/input/datasets/xusaiai/cifar-10-pythontargz', train=True, download=True, transform=transform_train
        )
        self.test_ds = torchvision.datasets.CIFAR10(
            root='/kaggle/input/datasets/xusaiai/cifar-10-pythontargz', train=False, download=False, transform=transform_test
        )

        self.train_dl = DataLoader(
            self.train_ds, batch_size=batch_size,
            shuffle=True, num_workers=num_workers, pin_memory=True
        )
        self.test_dl = DataLoader(
            self.test_ds, batch_size=batch_size,
            shuffle=False, num_workers=num_workers, pin_memory=True
        )

        # Fast label access (NO image loading)
        train_targets = self.train_ds.targets
        test_targets = self.test_ds.targets

        forget_train_idx = [i for i, y in enumerate(train_targets) if y == forget_class]
        retain_train_idx = [i for i, y in enumerate(train_targets) if y != forget_class]

        forget_test_idx = [i for i, y in enumerate(test_targets) if y == forget_class]
        retain_test_idx = [i for i, y in enumerate(test_targets) if y != forget_class]

        self.forget_train_dataset = Subset(self.train_ds, forget_train_idx)
        self.retain_train_dataset = Subset(self.train_ds, retain_train_idx)
        self.forget_test_dataset = Subset(self.test_ds, forget_test_idx)
        self.retain_test_dataset = Subset(self.test_ds, retain_test_idx)

        print(f"Forget Train: {len(self.forget_train_dataset)}")
        print(f"Retain Train: {len(self.retain_train_dataset)}")
        print(f"Forget Test : {len(self.forget_test_dataset)}")
        print(f"Retain Test : {len(self.retain_test_dataset)}")

    # ---------------------------------------------------------
    # TRAINING
    # ---------------------------------------------------------
    def train_model(
        self,
        num_classes=10,
        epochs=10,
        lr=5e-4,
        momentum=0.9,
        weight_decay=5e-4,
        save_path=None,
    ):
        if self.train_ds is None:
            self.load_data()
            
        weights =  ResNet18_Weights.IMAGENET1K_V1
        model = resnet18(weights=weights)
        '''model.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        model.maxpool = nn.Identity()'''
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.5),   # keeps MC dropout working
            nn.Linear(in_features, num_classes)
        )
        model.to(self.device)

        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )
        
        criterion = nn.CrossEntropyLoss()

        best_acc = 0.0
        best_state = None

        print("Training on CIFAR-10...")
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0

            for images, labels in tqdm(self.train_dl, desc=f"Epoch {epoch+1}/{epochs}"):
                images = images.to(self.device)
                labels = labels.to(self.device)

                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

            # Validation
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for images, labels in self.test_dl:
                    images = images.to(self.device)
                    labels = labels.to(self.device)
                    preds = model(images).argmax(dim=1)
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

            acc = 100.0 * correct / total
            print(
                f"Epoch {epoch+1} | "
                f"Loss {running_loss/len(self.train_dl):.4f} | "
                f"Acc {acc:.2f}%"
            )

            if acc > best_acc:
                best_acc = acc
                best_state = copy.deepcopy(model.state_dict())
                if save_path:
                    torch.save(best_state, save_path)

        model.load_state_dict(best_state)
        self.model = model

        print(f"Best Test Accuracy: {best_acc:.2f}%")
        return model

    # ---------------------------------------------------------
    # UNCERTAINTY (MC DROPOUT)
    # ---------------------------------------------------------
    @torch.no_grad()
    def compute_uncertainty(
        self,
        num_mc_samples=50,
        batch_size=128,
        method="predictive_entropy",  # or "predictive_entropy"
    ):
        if self.model is None:
            raise ValueError("Train model first")

        loader = DataLoader(
            self.forget_train_dataset,
            batch_size=batch_size,
            shuffle=False
        )

        self.model.eval()
        enable_mc_dropout(self.model)

        uncertainties = []

        for images, _ in tqdm(loader, desc="Computing uncertainty"):
            images = images.to(self.device)

            logits_mc = torch.stack(
                [self.model(images) for _ in range(num_mc_samples)],
                dim=0
            )  # [T, B, C]

            probs_mc = F.softmax(logits_mc, dim=-1)
            mean_probs = probs_mc.mean(dim=0)

            # Predictive entropy
            entropy_mean = -torch.sum(
                mean_probs * torch.log(mean_probs + 1e-8), dim=1
            )

            if method == "predictive_entropy":
                uncertainty = entropy_mean

            elif method == "mutual_information":
                entropy_mc = -torch.sum(
                    probs_mc * torch.log(probs_mc + 1e-8), dim=-1
                ).mean(dim=0)
                uncertainty = entropy_mean - entropy_mc

            else:
                raise ValueError("Invalid uncertainty method")

            uncertainties.extend(uncertainty.cpu().numpy())

        self.uncertainties = np.array(uncertainties)
        print(f"Uncertainty mean = {self.uncertainties.mean():.4f}")

    # ---------------------------------------------------------
    # CURRICULUM LOADER
    # ---------------------------------------------------------
    def get_forget_loader(self, mode="curriculum_high_first", batch_size=128):
        if self.uncertainties is None:
            raise ValueError("Run compute_uncertainty() first")

        idx = np.arange(len(self.forget_train_dataset))

        if mode == "normal":
            np.random.shuffle(idx)
        elif mode == "curriculum_high_first":
            idx = idx[np.argsort(-self.uncertainties)]
        elif mode == "curriculum_low_first":
            idx = idx[np.argsort(self.uncertainties)]
        else:
            raise ValueError("Invalid mode")

        return DataLoader(
            Subset(self.forget_train_dataset, idx),
            batch_size=batch_size,
            shuffle=False
        )

In [8]:
orderer = UncertaintyCurriculumOrderer(seed=42)
orderer.load_data(batch_size=64,forget_class=1)

Forget Train: 5000
Retain Train: 45000
Forget Test : 1000
Retain Test : 9000


In [9]:
model = orderer.train_model(epochs=5,save_path="./pretrained_resnet18_cifar10_5epochs.pt")

Training on CIFAR-10...


Epoch 1/5: 100%|██████████| 782/782 [02:44<00:00,  4.75it/s]


Epoch 1 | Loss 0.7737 | Acc 90.46%


Epoch 2/5: 100%|██████████| 782/782 [02:43<00:00,  4.77it/s]


Epoch 2 | Loss 0.3098 | Acc 92.74%


...

Epoch 3 | Loss 0.2308 | Acc 93.70%


Epoch 4/5: 100%|██████████| 782/782 [02:44<00:00,  4.76it/s]


Epoch 4 | Loss 0.1917 | Acc 94.39%


Epoch 5/5: 100%|██████████| 782/782 [02:44<00:00,  4.77it/s]


Epoch 5 | Loss 0.1608 | Acc 94.50%
Best Test Accuracy: 94.50%


In [10]:
orderer.compute_uncertainty(num_mc_samples=30,batch_size=128,method="predictive_entropy")

Computing uncertainty: 100%|██████████| 40/40 [02:28<00:00,  3.70s/it]

Uncertainty mean = 0.1310


In [11]:
batch_size = 16
num_workers = 4

retain_train_dl = DataLoader(orderer.retain_train_dataset, batch_size=batch_size,shuffle=True, num_workers=num_workers, pin_memory=True)
retain_test_dl = DataLoader(orderer.retain_test_dataset, batch_size=batch_size,shuffle=False, num_workers=num_workers, pin_memory=True)

forget_train_dl = DataLoader(orderer.forget_train_dataset, batch_size=batch_size,shuffle=True, num_workers=num_workers, pin_memory=True)
forget_test_dl = DataLoader(orderer.forget_test_dataset, batch_size=batch_size,shuffle=False, num_workers=num_workers, pin_memory=True)

### Before Unlearning

In [12]:
model.load_state_dict(torch.load('/kaggle/working/pretrained_resnet18_cifar10_5epochs.pt'))
print('Forget Test Accuracy:',evaluate(model, forget_test_dl, orderer.device))
print('Retain Test Accuracy:',evaluate(model, retain_test_dl, orderer.device))
print('MIA:',get_membership_attack_prob(retain_train_dl, forget_train_dl, orderer.test_dl,model))

Forget Test Accuracy: 96.9
Retain Test Accuracy: 94.23333333333333
MIA: 0.8352


In [13]:
forget_train_low2high_dl = orderer.get_forget_loader(mode="curriculum_low_first", batch_size=32)
forget_train_high2low_dl = orderer.get_forget_loader(mode="curriculum_high_first", batch_size=32)
forget_train_random_dl = orderer.get_forget_loader(mode="normal", batch_size=32)

In [14]:
#random ordering
model.load_state_dict(torch.load('/kaggle/working/pretrained_resnet18_cifar10_5epochs.pt'))
model  = gradient_ascent_unlearning(model,forget_train_random_dl,orderer.device,lr=5e-6,epochs=5)
print('[Random Order] Forget Test Accuracy:',evaluate(model, forget_test_dl, orderer.device))
print('[Random Order] Retain Test Accuracy:',evaluate(model, retain_test_dl, orderer.device))
print('[Random Order] MIA:',get_membership_attack_prob(retain_train_dl, forget_train_dl, orderer.test_dl,model))

#Low2High
model.load_state_dict(torch.load('/kaggle/working/pretrained_resnet18_cifar10_5epochs.pt'))
model  = gradient_ascent_unlearning(model,forget_train_low2high_dl,orderer.device,lr=5e-6,epochs=5)
print('[Low2High Order] Forget Test Accuracy:',evaluate(model, forget_test_dl, orderer.device))
print('[Low2High Order] Retain Test Accuracy:',evaluate(model, retain_test_dl, orderer.device))
print('[Low2High Order] MIA:',get_membership_attack_prob(retain_train_dl, forget_train_dl, orderer.test_dl,model))

#High2Low
model.load_state_dict(torch.load('/kaggle/working/pretrained_resnet18_cifar10_5epochs.pt'))
model  = gradient_ascent_unlearning(model,forget_train_high2low_dl,orderer.device,lr=5e-6,epochs=5)
print('[High2Low Order] Forget Test Accuracy:',evaluate(model, forget_test_dl, orderer.device))
print('[High2Low Order] Retain Test Accuracy:',evaluate(model, retain_test_dl, orderer.device))
print('[High2Low Order] MIA:',get_membership_attack_prob(retain_train_dl, forget_train_dl, orderer.test_dl,model))

[GA] Epoch 1/5 | Avg Ascent Loss: -6.3035
[GA] Epoch 2/5 | Avg Ascent Loss: -9.5608
[GA] Epoch 3/5 | Avg Ascent Loss: -13.0533
[GA] Epoch 4/5 | Avg Ascent Loss: -16.7130
[GA] Epoch 5/5 | Avg Ascent Loss: -20.4258
[Random Order] Forget Test Accuracy: 0.0
[Random Order] Retain Test Accuracy: 79.0
[Random Order] MIA: 0.227
[GA] Epoch 1/5 | Avg Ascent Loss: -6.2739
[GA] Epoch 2/5 | Avg Ascent Loss: -9.7277
[GA] Epoch 3/5 | Avg Ascent Loss: -13.2846
[GA] Epoch 4/5 | Avg Ascent Loss: -16.9516
[GA] Epoch 5/5 | Avg Ascent Loss: -20.7238
[Low2High Order] Forget Test Accuracy: 0.0
[Low2High Order] Retain Test Accuracy: 84.96666666666667
[Low2High Order] MIA: 0.1722
[GA] Epoch 1/5 | Avg Ascent Loss: -6.2915
[GA] Epoch 2/5 | Avg Ascent Loss: -9.7249
[GA] Epoch 3/5 | Avg Ascent Loss: -13.2972
[GA] Epoch 4/5 | Avg Ascent Loss: -16.9274
[GA] Epoch 5/5 | Avg Ascent Loss: -20.7004
[High2Low Order] Forget Test Accuracy: 0.0
[High2Low Order] Retain Test Accuracy: 76.03333333333333
[High2Low Order] MIA: 0